In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from torch.nn import *
from flash_attn import flash_attn_qkvpacked_func, flash_attn_kvpacked_func
from IPython.display import clear_output
torch._dynamo.config.cache_size_limit = 64
class Plot(dict):
    t = 1
    def __init__(s, n=100, m=1000):
        s.__dict__ |= locals()
    def __call__(s, **_):
        for i,_ in _.items():
            si = s.get(i)
            if not si:
                s[i] = si = [torch.zeros(1024,*_.shape,device=_.device),0,0]
            if si[1] != s.t//s.n:
                si[0][si[1]] /= si[2]
                si[1] = s.t//s.n
                si[2] = 0
            si[0][s.t//s.n] += _
            si[2] += 1
        s.t += 1
        if not s.t%s.m:
            s.show()
    def show(s):
        for j,(i,_) in enumerate(s.items(),1):
            plt.subplot(1,len(s),j).plot(_[0][:_[1]].cpu())
        clear_output(True)
        plt.show()
class Res(Sequential):
    def forward(s, _):
        return super().forward(_) + _
class T(Sequential):
    def forward(s, _):
        return super().forward(_.mT).mT
class F(Module):
    def __init__(s, _):
        super().__init__()
        s._ = _
    def forward(s, _):
        return s._(_)
class Attn(Linear):
    def __init__(s, i, h=0, G=1, g=1, r=(7,7), drop=0, alibi=1., p=0):
        h = h or i
        super().__init__(i, G*(g+2)*h)
        s._ = Linear(h if p else G*g*h, i, 0)
        s.__dict__ |= locals()
    def forward(s, _):
        _ = super().forward(_)
        _ = (flash_attn_qkvpacked_func if s.g<2 else flash_attn_kvpacked_func)(
            *[_.unflatten(-1,(3,-1,s.h))]if s.g<2 else[_[...,:-2*s.G*s.h].unflatten(-1,(-1,s.h)),_[...,-2*s.G*s.h:].unflatten(-1,(2,-1,s.h))],
            dropout_p=s.drop,
            window_size=s.r,
            alibi_slopes=torch.arange(s.G*s.g*s.alibi, device='cuda'),
        )
        return s._(_.norm(s.p,-2)if s.p else _.flatten(-2))
class SwiGLU(Linear):
    def __init__(s, i, h=0):
        super().__init__(i, 2*(h or i))
        s._ = Linear(h or i, i, 0)
    def forward(s, _):
        return s._(functional.silu(torch.mul(*super().forward(_).chunk(2,-1))))
def At(i, *_):
    return Res(RMSNorm(i), Attn(i, *_))
def Ff(i, *_):
    return Res(RMSNorm(i), SwiGLU(i, *_))
def U(i, *_):
    return Res(
        Linear(i, i, 0),
        T(MaxPool1d(2)),
        *_,
        Linear(i, i*2),
        F(lambda _:functional.pad(_.unflatten(-1,(2,-1)).flatten(-3,-2)[...,:-1,:],(0,0,1,0))),
    )
A = torch.from_numpy(np.memmap(f'../../py.txt', 'B', 'r'))
A.as_strided_((len(A)//4096-1,8193),(4096,1));

In [ ]:
i,h,G,g,a = 256,16,16,1,1024
m = Sequential(
    Embedding(256, i),
    At(i, h, G, g), Ff(i, a),
    U(i,
        At(i, h, G, g), Ff(i, a),
        U(i,
            At(i, h, G, g), Ff(i, a),
            U(i,
                At(i, h, G, g), Ff(i, a),
                U(i,
                    At(i, h, G, g), Ff(i, a),
                    U(i,
                        At(i, h, G, g), Ff(i, a),
                    ),
                    At(i, h, G, g), Ff(i, a),
                ),
                At(i, h, G, g), Ff(i, a),
            ),
            At(i, h, G, g), Ff(i, a),
        ),
        At(i, h, G, g), Ff(i, a),
    ),
    At(i, h, G, g), Ff(i, a),
    Linear(i, 256)
).to('cuda',torch.bfloat16)
m = torch.compile(m, dynamic=False)
o = torch.optim.Adam(m.parameters(), eps=1e-15)
O = torch.optim.lr_scheduler.OneCycleLR(o, .001, 100000, pct_start=.01, div_factor=10, final_div_factor=10)
plot = Plot()
while O.last_epoch < O.total_steps:
    t = A[torch.randint(len(A),(8,))].to('cuda',torch.long)
    x = t[:,:-1]
    t = t[:,1:]
    x = x.put(torch.randint(x.numel(),[x.numel()*3//4],device='cuda'),torch.randint(256,[x.numel()*3//4],device='cuda'))
    y = m(x)
    l = functional.cross_entropy(y.mT, t)
    l.backward()
    o.step()
    o.zero_grad()
    O.step()
    with torch.no_grad():
        plot(l=l, a=(y.argmax(2)==t).sum()/t.numel()) #-expm1((np.log((256-1)/(256-2))+log1p(-2*p)-log1p(-p))*(256-1)))**(256-1), (-expm1(log1p(-p)*(256-1)))**(256-1), exp(-log1p((1-p)/p/(256-1))*(256-1)))))

In [ ]:
i,h,G,g,a = 256,32,8,1,1024
m = Sequential(
    Embedding(256, i),
    At(i, h, G, g), Ff(i, a),
    U(i,
        At(i, h, G, g), Ff(i, a),
        U(i,
            At(i, h, G, g), Ff(i, a),
            U(i,
                At(i, h, G, g), Ff(i, a),
                U(i,
                    At(i, h, G, g), Ff(i, a),
                    U(i,
                        At(i, h, G, g), Ff(i, a),
                    ),
                    At(i, h, G, g), Ff(i, a),
                ),
                At(i, h, G, g), Ff(i, a),
            ),
            At(i, h, G, g), Ff(i, a),
        ),
        At(i, h, G, g), Ff(i, a),
    ),
    At(i, h, G, g), Ff(i, a),
    Linear(i, 256)
).to('cuda',torch.bfloat16)
m = torch.compile(m, dynamic=False)
o = torch.optim.Adam(m.parameters(), eps=1e-15)
O = torch.optim.lr_scheduler.OneCycleLR(o, .001, 100000, pct_start=.01, div_factor=10, final_div_factor=10)
plot = Plot()
while O.last_epoch < O.total_steps:
    t = A[torch.randint(len(A),(8,))].to('cuda',torch.long)
    x = t[:,:-1]
    t = t[:,1:]
    x = x.put(torch.randint(x.numel(),[x.numel()*3//4],device='cuda'),torch.randint(256,[x.numel()*3//4],device='cuda'))
    y = m(x)
    l = functional.cross_entropy(y.mT, t)
    l.backward()
    o.step()
    o.zero_grad()
    O.step()
    with torch.no_grad():
        plot(l=l, a=(y.argmax(2)==t).sum()/t.numel()) #-expm1((np.log((256-1)/(256-2))+log1p(-2*p)-log1p(-p))*(256-1)))**(256-1), (-expm1(log1p(-p)*(256-1)))**(256-1), exp(-log1p((1-p)/p/(256-1))*(256-1)))))

In [ ]:
i,h,G,g,a = 256,64,4,1,1024
m = Sequential(
    Embedding(256, i),
    At(i, h, G, g), Ff(i, a),
    U(i,
        At(i, h, G, g), Ff(i, a),
        U(i,
            At(i, h, G, g), Ff(i, a),
            U(i,
                At(i, h, G, g), Ff(i, a),
                U(i,
                    At(i, h, G, g), Ff(i, a),
                    U(i,
                        At(i, h, G, g), Ff(i, a),
                    ),
                    At(i, h, G, g), Ff(i, a),
                ),
                At(i, h, G, g), Ff(i, a),
            ),
            At(i, h, G, g), Ff(i, a),
        ),
        At(i, h, G, g), Ff(i, a),
    ),
    At(i, h, G, g), Ff(i, a),
    Linear(i, 256)
).to('cuda',torch.bfloat16)
m = torch.compile(m, dynamic=False)
o = torch.optim.Adam(m.parameters(), eps=1e-15)
O = torch.optim.lr_scheduler.OneCycleLR(o, .001, 100000, pct_start=.01, div_factor=10, final_div_factor=10)
plot = Plot()
while O.last_epoch < O.total_steps:
    t = A[torch.randint(len(A),(8,))].to('cuda',torch.long)
    x = t[:,:-1]
    t = t[:,1:]
    x = x.put(torch.randint(x.numel(),[x.numel()*3//4],device='cuda'),torch.randint(256,[x.numel()*3//4],device='cuda'))
    y = m(x)
    l = functional.cross_entropy(y.mT, t)
    l.backward()
    o.step()
    o.zero_grad()
    O.step()
    with torch.no_grad():
        plot(l=l, a=(y.argmax(2)==t).sum()/t.numel()) #-expm1((np.log((256-1)/(256-2))+log1p(-2*p)-log1p(-p))*(256-1)))**(256-1), (-expm1(log1p(-p)*(256-1)))**(256-1), exp(-log1p((1-p)/p/(256-1))*(256-1)))))

In [ ]:
l = a = 0
with torch.no_grad():
    for t in range(100):
        t = torch.tensor(A[np.random.randint(0,len(A),8,'I')], dtype=torch.long, device='cuda')
        x = t.put(torch.randint(t.numel(),[t.numel()//2],device='cuda'),torch.randint(256,[t.numel()//2],device='cuda'))
        y = m(x)
        t = LigerFusedLinearCrossEntropyFunction.apply(y.flatten(0,1), getattr(m,'0').weight, t.ravel())[0]
        l += t
        a += entropy((-t).float().exp(), 256)
print(l, a)

In [ ]:
import matplotlib.pyplot as plt
class MyAdd(Add):
    _ = []
    def forward(s, _):
        _ = [s(_) for s in s]
        s._ += [_.norm() for _ in _],
        return reduce(torch.add, _)
class MyRes(Res):
    _ = []
    def forward(s, _):
        t = super().forward(_)
        s._ += [_.norm(), t.norm()],
        return _ + t
for _ in m.modules():
    if isinstance(_, Add):
        _.__class__ = MyAdd
    elif isinstance(_, Res):
        _.__class__ = MyRes
with torch.no_grad():
    m._orig_mod(x)
plt.plot(torch.tensor(MyAdd._).to('cpu', torch.float))
plt.yscale('log')

In [ ]:
plt.plot(torch.tensor(MyRes._).to('cpu',torch.float))
plt.yscale('log')